# Hasan 2017: 공개 FEM 곡선과 현재 HCF 모세관 모델 비교
24개 표본을 확인합니다. 서로 다른 물리 모델의 차이이며 COMSOL/실제 ARF 정확도 인증이 아닙니다.
[보고서](https://github.com/kimheeseo/LSCNS/tree/main/COMSOL/public_reference_benchmarks/hasan2017) · [원 논문](https://arxiv.org/abs/1708.06879v2)
±는 그림 읽기 범위이며 통계적 신뢰구간이 아닙니다. 단위 fs²/cm.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('https://raw.githubusercontent.com/kimheeseo/LSCNS/main/COMSOL/public_reference_benchmarks/hasan2017/comparison_24.csv')
display(df[['tubes','wavelength_um','published_FEM_GVD_estimate_fs2_cm','raster_reading_halfwidth_fs2_cm','current_capillary_FEM_GVD_fs2_cm','nominal_absolute_difference_pct']])
fig, axes = plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax,(n,g) in zip(axes,df.groupby('tubes')):
    ax.errorbar(g.wavelength_um,g.published_FEM_GVD_estimate_fs2_cm,yerr=g.raster_reading_halfwidth_fs2_cm,fmt='o',label='Published FEM (digitized)')
    ax.plot(g.wavelength_um,g.current_capillary_FEM_GVD_fs2_cm,label='Current ideal capillary FEM')
    ax.set(title=f'{n} tubes',xlabel='Wavelength (um)')
    ax.grid(alpha=.2)
axes[0].set_ylabel('GVD (fs^2/cm)')
axes[0].legend()
plt.tight_layout()
plt.show()

## 선택: 원 논문에서 다시 추출하고 FEM 재실행
아래 셀은 공개 PDF와 고정된 기존 솔버를 내려받아 해시를 확인하고 다시 계산합니다. 이 노트북은 이번 작업에서 Google Colab 화면으로 실행하지 않았습니다. 재계산 스크립트는 Python 3.12 환경에서 실행했습니다.

In [ ]:
import urllib.request, hashlib, pathlib, subprocess, sys, tempfile
subprocess.check_call([sys.executable,'-m','pip','-q','install','numpy','scipy','pillow','pymupdf'])
folder=pathlib.Path(tempfile.mkdtemp(prefix='hasan_comparison_'))
b=urllib.request.urlopen('https://raw.githubusercontent.com/kimheeseo/LSCNS/main/COMSOL/public_reference_benchmarks/hasan2017/compare.py',timeout=60).read()
assert hashlib.sha256(b).hexdigest() == 'f5631afde6428573f4cefa610e46ad4f5909167187d26fd42915ecf1f39e606e', 'Script changed: inspect before rerunning'
(folder/'compare.py').write_bytes(b)
subprocess.check_call([sys.executable,str(folder/'compare.py')])
display(pd.read_csv(folder/'comparison_24.csv'))